# Market Sentiment Model
In this notebook we are going to explore building a transformer model to analyze overall market sentiment based off of current
news articles.

In [1]:
import kagglehub
import numpy as np
import pandas as pd
from torch import nn
from torch.optim import lr_scheduler
from torch.utils.data import TensorDataset, DataLoader

from src.my_engine.config import ModelConfig, TrainerConfig
from src.my_engine.trainer import Trainer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [2]:
# Download latest version
path = kagglehub.dataset_download("sbhatti/financial-sentiment-analysis")

print("Path to dataset files:", path)

df = pd.read_csv(f"{path}/data.csv")
df['Sentiment'] = df['Sentiment'].astype('category')
df['Sentiment'] = df['Sentiment'].cat.codes
print(df.shape)
df.head()

Path to dataset files: /home/alexsearle/.cache/kagglehub/datasets/sbhatti/financial-sentiment-analysis/versions/4
(5842, 2)


,Sentence,Sentiment
0,The GeoSolutions technology will leverage Bene...,2
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",0
2,"For the last quarter of 2010 , Componenta 's n...",2
3,According to the Finnish-Russian Chamber of Co...,1
4,The Swedish buyout firm has sold its remaining...,1


In [3]:
X_train, X_test, y_train, y_test = train_test_split(df['Sentence'], df['Sentiment'], test_size=0.2, random_state=42)

In [14]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
train_text_tokens = [str(text).lower() for text in X_train]
test_text_tokens = [str(text).lower() for text in X_test]
train_encoding = tokenizer(train_text_tokens, return_tensors="pt", padding=True, truncation=True)
test_encoding = tokenizer(test_text_tokens, return_tensors="pt", padding=True, truncation=True)

train_tokens = train_encoding["input_ids"]
train_attention_mask = train_encoding["attention_mask"]
train_labels = torch.tensor(y_train.values, dtype=torch.long)

test_tokens = test_encoding["input_ids"]
test_attention_mask = test_encoding["attention_mask"]
test_labels = torch.tensor(y_test.values, dtype=torch.long)

train_ds = TensorDataset(train_tokens, train_attention_mask, train_labels)
test_ds = TensorDataset(test_tokens, test_attention_mask, test_labels)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=True)

In [15]:
num_epochs = 3
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5, weight_decay=1e-3)
model.train()
for epoch in range(num_epochs):
    loss_batch = 0.0
    accuracy_batch = 0.0
    count = 0

    for X, mask, y in tqdm(train_loader):
       X, mask, y = X.to(device), mask.to(device), y.to(device)
       optimizer.zero_grad()

       output = model(input_ids=X, attention_mask=mask, labels=y)
       logits = output.logits
       loss = output.loss
       loss.backward()
       optimizer.step()

       loss_batch += loss.item() * X.size(0)
       accuracy_batch += (logits.argmax(dim=1) == y).sum().item()
       count += X.size(0)

    model.eval()
    val_loss = 0.0
    val_accuracy = 0.0
    val_count = 0

    for X, mask, y in tqdm(test_loader):
        X, mask, y = X.to(device), mask.to(device), y.to(device)
        with torch.no_grad():
            output = model(input_ids=X, attention_mask=mask, labels=y)
            logits = output.logits
            loss = output.loss

            preds = logits.argmax(dim=1)

            val_loss += loss.item() * X.size(0)
            val_accuracy += (preds == y).sum().item()
            val_count += X.size(0)

    print(f"Epoch {epoch}:")
    print(f"\tTRAIN: Loss: {loss_batch/count:.4f}, Accuracy: {accuracy_batch/count:.2%}")
    print(f"\tTEST: loss: {val_loss/val_count:.4f}, Accuracy: {val_accuracy/val_count:.2f}")

100%|██████████| 19/19 [00:01<00:00, 13.19it/s]


Epoch 0:
	TRAIN: Loss: 0.8338, Accuracy: 61.67%
	TEST: loss: 0.5849, Accuracy: 0.74


100%|██████████| 19/19 [00:01<00:00, 13.29it/s]


Epoch 1:
	TRAIN: Loss: 0.4545, Accuracy: 79.39%
	TEST: loss: 0.4398, Accuracy: 0.80


100%|██████████| 19/19 [00:01<00:00, 12.79it/s]

Epoch 2:
	TRAIN: Loss: 0.3172, Accuracy: 85.66%
	TEST: loss: 0.4411, Accuracy: 0.80


In [16]:
torch.save(model.state_dict(), "../models/sentiment.pt")